# Modeling

Packages and setup

In [ ]:
# Imports & settings
from foodcast.imports import *
notebook_settings() 
os.chdir(PROJECT_ROOT)
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_4, DATA_DIR_3_x = return_dir()
DATA_DIR_3_1, DATA_DIR_3_2, DATA_DIR_3_3, DATA_DIR_3_4, DATA_DIR_3_5, DATA_DIR_3_6, DATA_DIR_3_7, DATA_DIR_3_8, _ = DATA_DIR_3_x
from foodcast.tools.rolling import rolling_window_avg, add_interday_variables, add_intraday_variables, unroll, season_from_month
from foodcast.tools.takeout import takeout

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
data = load_all_res_3_4_ai()

# Load special data
customers_before_after = pd.read_pickle(DATA_DIR_3 / 'customers_before_after.pkl')
grouping_mappings = pd.read_pickle(DATA_DIR_3 / 'grouping_mappings.pkl')
animal_categories_items = pd.read_pickle(DATA_DIR_3 / 'animal_categories_items.pkl')

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=pd.errors.PerformanceWarning)

    totals = [0, 0, 0]
    for loc_id in location_ids_by_coverage:
        totals[0] += data[loc_id].shape[0]

    df_all_list = []   
    for loc_id in location_ids_by_coverage:
        df = (
            data[loc_id]
            .query('~is_drink')
            .query('~is_nonfood')
            .query('~is_nonmeal_merchandise'))

        totals[1] += df.shape[0]
        filepath = Path(DATA_DIR_3_5) / f'{loc_id}.parquet'
        #if not filepath.exists():
        df.to_parquet(filepath, index=False)

        df = (
            df
            .fillna({'item_modifications':''})
            .query('~item_modifications.str.lower().str.contains(@takeout)')
            .query('~item_name.str.lower().str.contains(@takeout)'))
        
        totals[2] += df.shape[0]
        filepath = Path(DATA_DIR_3_6) / f'{loc_id}.parquet'
        #if not filepath.exists():
        df.to_parquet(filepath, index=False)

In [ ]:
loc_id = location_ids_by_coverage[2]
menu_labels = pd.read_csv(Path('scripts')/'labeling'/'dish_labels'/f'{loc_id}.csv')
old = data[loc_id].item_name.value_counts().to_frame(name='count').reset_index()
new = menu_labels.item_name
pd.merge(old, new, on='item_name', how='left', indicator=True).query('_merge == "left_only"')